# PWV03d : Two-Point Temporal Correlation Function — per filter (Sylvie's own estimator)

## Overview

This notebook is a variant of `PWV03_TwoPoint_TemporalCorrelation_separateFilters.ipynb`
that implements the two-point temporal correlation function using
**Sylvie Dagoret-Campagne's own estimator**, originally coded in:

- `OLD_PWV_FrequencyFitGPAndMerra2ComparisonMarch2025.ipynb`
- `OLD_PWV_FrequencyFitSinusAndMerra2ComparisonMarch2025.ipynb`

## Statistical estimator

The discrete correlation function is estimated as:
$$
C(\Delta t) =
  \frac{\bigl\langle \delta\mathrm{PWV}(t_i)\,\delta\mathrm{PWV}(t_j)\bigr\rangle_{|\Delta t_{ij} \in \mathrm{bin}|} }
       {\sigma^2_{\mathrm{PWV}}}
\qquad\text{where}\quad \Delta t_{ij} = t_j - t_i > 0
$$

The implementation (`ComputePWVAndTimeDifference`) loops over all ordered pairs
$(i, j)$ with $t_j \geq t_i$, stores their time lag and PWV product, and then
`ComputeMyDCF_PwvixPwvj` bins the products in a user-defined lag grid and
normalises by the global $\sigma^2$.

## Differences with PWV03 reference notebook

- Sections 1–3 (data loading, quality cuts, night-mean subtraction) are **identical**.
- Section 4 uses Sylvie's explicit pair loop + `ComputeMyDCF_PwvixPwvj` binning.
- The nightly mean subtraction now mirrors the per-night approach of PWV03.
- The same exponential fit and per-filter analysis are kept.

## Key design choices carried over from OLD notebooks

- The **mean PWV** used for the normalisation can be taken either globally
  (as in the OLD notebooks) or per-night (as in PWV03); this notebook provides
  both approaches for comparison, defaulting to the per-night subtraction.
- Pairs are **not restricted to the same night** in `ComputePWVAndTimeDifference`:
  any pair with $\Delta t > 0$ is included.  A same-night variant is also provided.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from astropy.time import Time
from scipy.optimize import curve_fit

%matplotlib inline

# ── publication rc params ─────────────────────────────────────────────────────
mpl.rcParams.update({
    'figure.dpi'     : 150,
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 13,
    'axes.titlesize' : 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.linewidth' : 1.1,
})


In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()


In [ ]:
pathfigs = 'figs_PWV03dcorr'
prefix   = 'pwv03dcorr_sylvie'
os.makedirs(pathfigs, exist_ok=True)
figtype  = '.pdf'


In [ ]:
FLAG_WITHCOLLIMATOR     = True
datetime_WITHCOLLIMATOR = pd.to_datetime('2023-09-30 00:00:00.0+0000')


In [ ]:
SIGMA_repeat = 0.05

---
## 1. Load data & build Time column


👉 ça simule l’ancien module

⚠️ marche souvent, mais pas garanti si d’autres modules ont bougé

In [ ]:
# hack pour la lecture des pickes
import numpy as np
import types
import sys

sys.modules['numpy.rec'] = np.rec

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split("/")[-1]
print(f"Loading: {atmfilename}")

if "parquet" in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec  = pd.DataFrame(specdata)

# Time from DATE-OBS (same as PWV01/02)
df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"], utc=True)
df_spec = df_spec.sort_values("Time", ascending=True).reset_index(drop=True)


if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec["Time"] > datetime_WITHCOLLIMATOR].copy()
    

print(f"Shape: {df_spec.shape}")
print(f"Date range: {df_spec['Time'].min().date()}  ->  {df_spec['Time'].max().date()}")

In [ ]:
## Re-compute nightObs from Time (id is buggy in Corentin's files)

In [ ]:
df_spec["nightObs"] = df_spec.apply(lambda x: x['id']//100_000, axis=1)
df_spec["seq_num"]  = df_spec["id"] % 100_000
df_spec["night_from_time_utc"] = (
    df_spec["Time"].dt.strftime("%Y%m%d").astype(int)
)

mismatch = df_spec[
    df_spec["night_from_time_utc"] != df_spec["nightObs"]
]

len(mismatch), len(df_spec)

In [ ]:
FLAG_CORRECT_NIGHTOBS_VARIABLES = True

if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    print("COMPUTE NIGHTOBS FROM Time") 
    # Convertir au Chili
    df_spec["Time_local"] = df_spec["Time"].dt.tz_convert("America/Santiago")

    # Decalage de 12h vers le passe pour rattacher les obs apres minuit a la bonne nuit
    df_spec["nightObs_corr"] = (
        (df_spec["Time_local"] - pd.Timedelta(hours=12))
        .dt.strftime("%Y%m%d")
        .astype(int)
    )
if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    span = (
        df_spec.groupby("nightObs_corr")["Time"].max()
        - df_spec.groupby("nightObs_corr")["Time"].min()
    )
    print(span.max())
if FLAG_CORRECT_NIGHTOBS_VARIABLES and "run2026_v02" in version_run:
    df_spec["nightObs"] = df_spec["nightObs_corr"]

In [ ]:
if "run2026_v02" in version_run:
    df_spec["chi2_ram"] = df_spec["CHI2_FIT"]
    df_spec["chi2_rum"] = df_spec["CHI2_FIT"]

if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec["FILTER"].isin(PWV_FILTER_LIST)].copy()

pwv_cols = ["PWV [mm]_ram", "PWV [mm]_rum", "PWV [mm]_err_ram", "PWV [mm]_err_rum"]
df_spec.dropna(subset=pwv_cols, inplace=True)

print(f"After filter selection: {len(df_spec)} rows")

In [ ]:
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="CHI2_FIT", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_ram", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_rum", ext="norm")

---
## 2. Quality cuts


In [ ]:
FLAG_LOOSE_CUTS = True
FLAG_TIGHT_CUTS = False

In [ ]:
pathdata      = "data_PWV01seas"   # reuse cuts from PWV01 notebook
if FLAG_LOOSE_CUTS:
    filename_cuts = f"{pathdata}/cuts_loose_finaldecision.json" 
    cutstype_name = "loose-cuts"
elif FLAG_TIGHT_CUTS: 
    filename_cuts = f"{pathdata}/cuts_tight_finaldecision.json" 
    cutstype_name = "tight-cuts"
else:
    filename_cuts = f"{pathdata}/cuts_finaldecision.json" 
    cutstype_name = "standard-cuts"

cuts           = ParameterCutTools.load_cuts_json(filename_cuts)
list_of_params = list(cuts.keys())

selector    = ParameterCutSelection(df_spec, params=list_of_params, id_col="id")
flags       = selector.apply_cuts(cuts)
df_selected = df_spec.merge(flags, on="id")
df_keep     = df_selected[df_selected["pass_all_cuts"]].copy()

print(f"Tight cuts: {len(df_keep)} / {len(df_spec)} kept")

---
## 3. Prepare PWV time series — night-by-night mean subtraction


In [ ]:
# ── Best-estimate PWV ─────────────────────────────────────────────────────────
df_keep["PWV"]     = df_keep["PWV [mm]_ram"]
df_keep["PWV_err"] = df_keep["PWV [mm]_err_ram"]

# ── Time in minutes since first observation ───────────────────────────────────
t0_min = df_keep["MJD"].min() * 1440.0
df_keep["t_min"] = df_keep["MJD"] * 1440.0 - t0_min

# ── Night-by-night mean subtraction ──────────────────────────────────────────
# Compute the mean PWV for each nightObs separately, then broadcast back.
night_means = df_keep.groupby("nightObs")["PWV"].transform("mean")
df_keep["PWV_night_mean"] = night_means
df_keep["dPWV"]           = df_keep["PWV"] - night_means

# Pooled intra-night variance (used as C(0) denominator for all pairs)
pwv_var = df_keep["dPWV"].var(ddof=1) + SIGMA_repeat**2

# ── Diagnostics ───────────────────────────────────────────────────────────────
n_nights = df_keep["nightObs"].nunique()
night_stats = df_keep.groupby("nightObs").agg(
    n_obs     = ("PWV", "count"),
    pwv_mean  = ("PWV", "mean"),
    pwv_std   = ("PWV", "std"),
    dpwv_std  = ("dPWV", "std"),
)

print(f"N observations              : {len(df_keep)}")
print(f"N nights (nightObs)         : {n_nights}")
print(f"Median obs per night        : {night_stats['n_obs'].median():.0f}")
print()
print(f"Night-mean PWV  — range     : {night_stats['pwv_mean'].min():.2f} – "
      f"{night_stats['pwv_mean'].max():.2f} mm")
print(f"Night-mean PWV  — std across nights : "
      f"{night_stats['pwv_mean'].std():.3f} mm  (inter-night scatter)")
print()
print(f"Intra-night std — median per night  : "
      f"{night_stats['dpwv_std'].median():.3f} mm")
print(f"Pooled intra-night variance  sigma^2 : {pwv_var:.4f} mm^2  "
      f"=> sigma = {np.sqrt(pwv_var):.3f} mm")
print(f"Time span                   : {df_keep['t_min'].max()/1440:.1f} days  "
      f"({df_keep['t_min'].max()/60:.0f} hours)")

display(night_stats.describe().round(3))

In [ ]:
# ── Quick sanity plot: inter-night vs intra-night variability ─────────────────
fig0, axes0 = plt.subplots(1, 2, figsize=(14, 4))

# Left: nightly mean PWV over time
night_order = night_stats.sort_index()
axes0[0].errorbar(
    range(len(night_order)),
    night_order["pwv_mean"],
    yerr=night_order["pwv_std"],
    fmt="o", color="steelblue", ms=4, elinewidth=0.8, capsize=2,
    label="Nightly mean +/- std"
)
axes0[0].set_xlabel("Night index (chronological)", fontsize=12)
axes0[0].set_ylabel("PWV [mm]", fontsize=12)
axes0[0].set_title("Nightly mean PWV — inter-night variability", fontsize=11)
axes0[0].legend(fontsize=9)

# Right: histogram of dPWV (intra-night residuals)
axes0[1].hist(df_keep["dPWV"].dropna(), bins=60, color="steelblue",
              alpha=0.8, edgecolor="white", linewidth=0.3)
axes0[1].axvline(0, color="black", ls="--", lw=1)
axes0[1].set_xlabel(r"$\delta\mathrm{PWV}$ [mm]", fontsize=12)
axes0[1].set_ylabel("Count", fontsize=12)
axes0[1].set_title(
    rf"Intra-night residuals  $\sigma = {np.sqrt(pwv_var):.3f}$ mm",
    fontsize=11
)

fig0.suptitle(
    f"Night-by-night mean subtraction — {version_run}",
    fontsize=12, y=1.02
)
fig0.tight_layout()
figfile0 = f"{pathfigs}/{prefix}_{version_run}_night_mean_subtraction{figtype}"
fig0.savefig(figfile0, bbox_inches="tight")
print(f"Saved: {figfile0}")
plt.show()

---
## 4. Sylvie's estimator — pair loop and DCF binning

### `ComputePWVAndTimeDifference`

This function (ported from the OLD notebooks) iterates over **all ordered pairs**
$(i, j)$ with $\Delta t_{ij} = t_j - t_i \geq 0$ and returns:

- `all_DT`       — array of time lags $\Delta t$ (in the time unit of the input)
- `all_DPWV`     — array of PWV differences $\mathrm{PWV}_j - \mathrm{PWV}_i$
- `all_pwvpwv`   — array of products $(\mathrm{PWV}_i - \bar{\mathrm{PWV}})(\mathrm{PWV}_j - \bar{\mathrm{PWV}})$
- `all_PWVpairs` — array of $(\mathrm{PWV}_i, \mathrm{PWV}_j)$ pairs

The global (or per-night) mean $\bar{\mathrm{PWV}}$ is subtracted internally.

### `ComputeMyDCF_PwvixPwvj`

This function bins the pair products in a user-supplied array of time-bin edges
and computes the mean and standard error of the normalised product
$\langle P_i P_j \rangle / \sigma^2$ in each bin.

### Same-night variant

`ComputePWVAndTimeDifference_SameNight` is a new variant that restricts pairs
to the same `nightObs`, following the approach of the reference PWV03 notebook.


In [ ]:
def ComputePWVAndTimeDifference(df, meanpwv=None):
    """
    Compute all ordered pairs (i, j) with dt_ij = t_j - t_i >= 0.

    Original implementation: Sylvie Dagoret-Campagne (2025-03-14).
    Ported from OLD_PWV_FrequencyFitGPAndMerra2ComparisonMarch2025.ipynb.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns 't' (time, any unit) and 'pwv' (PWV in mm).
    meanpwv : float or None
        Mean PWV for subtraction. If None, the global mean of df['pwv'] is used.

    Returns
    -------
    all_DT, all_DPWV, all_pwvpwv, all_PWVpairs : np.ndarray
        Arrays of (time lag, PWV difference, product of deviations, PWV pair values).
    """
    all_DPWV     = []
    all_DT       = []
    all_pwvpwv   = []
    all_PWVpairs = []

    if meanpwv is None:
        meanpwv = df['pwv'].mean()

    # Vectorised implementation (faster than the original double loop)
    t_arr   = df['t'].values
    pwv_arr = df['pwv'].values
    dpwv_arr = pwv_arr - meanpwv
    N = len(t_arr)

    for i in range(N):
        for j in range(N):
            dt = t_arr[j] - t_arr[i]
            if dt >= -1e-9:   # t_j >= t_i (allow tiny floating-point tolerance)
                all_DT.append(dt)
                all_DPWV.append(pwv_arr[j] - pwv_arr[i])
                all_pwvpwv.append(dpwv_arr[i] * dpwv_arr[j])
                all_PWVpairs.append([pwv_arr[i], pwv_arr[j]])

    return (np.array(all_DT), np.array(all_DPWV),
            np.array(all_pwvpwv), np.array(all_PWVpairs))


In [ ]:
def ComputePWVAndTimeDifference_vectorized(df, meanpwv=None):
    """
    Vectorized version of ComputePWVAndTimeDifference (uses numpy triu_indices).
    Much faster for large datasets — recommended for production use.

    Includes both i<j (dt>0) and i>j (dt<0) pairs for forward lags only,
    i.e. only upper-triangle pairs (j > i in the sorted array).
    """
    if meanpwv is None:
        meanpwv = df['pwv'].mean()

    t_arr    = df['t'].values
    pwv_arr  = df['pwv'].values
    #dpwv_arr = pwv_arr - meanpwv
    dpwv_arr = pwv_arr

    # Upper triangle: all pairs (i, j) with j >= i
    ii, jj = np.triu_indices(len(t_arr), k=0)
    all_DT       = t_arr[jj]    - t_arr[ii]
    all_DPWV     = pwv_arr[jj]  - pwv_arr[ii]
    all_pwvpwv   = dpwv_arr[ii] * dpwv_arr[jj]
    all_PWVpairs = np.column_stack([pwv_arr[ii], pwv_arr[jj]])

    return all_DT, all_DPWV, all_pwvpwv, all_PWVpairs


In [ ]:
def ComputePWVAndTimeDifference_SameNight(df_full, meanpwv_col='dPWV', t_col='t_min',
                                            night_col='nightObs', sigma2=None):
    """
    Same-night variant: only pairs (i, j) belonging to the same nightObs.

    This matches the approach of the reference PWV03 notebook and avoids
    cross-night contamination.

    Parameters
    ----------
    df_full      : pd.DataFrame — must contain t_col, meanpwv_col, night_col.
    meanpwv_col  : str — column of nightly-mean-subtracted PWV residuals.
    t_col        : str — time column (in minutes by default).
    night_col    : str — night identifier column.
    sigma2       : float or None — normalisation variance; if None, computed from data.

    Returns
    -------
    df_pairs : pd.DataFrame with columns ['dt', 'DPWV', 'PwvixPwvj']
    sigma2   : float — normalisation variance used
    """
    dt_list, dpwv_list, prod_list = [], [], []

    for night in sorted(df_full[night_col].unique()):
        sub = df_full[df_full[night_col] == night]
        if len(sub) < 2:
            continue
        t_n  = sub[t_col].values
        dp_n = sub[meanpwv_col].values

        ii, jj = np.triu_indices(len(t_n), k=1)
        dt_n   = t_n[jj] - t_n[ii]       # always >= 0 (upper triangle, k=1)
        prod_n = dp_n[ii] * dp_n[jj]

        dt_list.append(dt_n)
        dpwv_list.append(dp_n[jj] - dp_n[ii])
        prod_list.append(prod_n)

    df_pairs = pd.DataFrame({
        'dt'        : np.concatenate(dt_list),
        'DPWV'      : np.concatenate(dpwv_list),
        'PwvixPwvj' : np.concatenate(prod_list),
    })

    if sigma2 is None:
        sigma2 = df_full[meanpwv_col].var(ddof=1) + SIGMA_repeat**2

    return df_pairs, sigma2


In [ ]:
def ComputeMyDCF_PwvixPwvj(df_pairs, list_of_bins, averPWV, sigPWV):
    """
    Compute the average discrete correlation function from pair products.

    Original implementation: Sylvie Dagoret-Campagne (2025-03-14).
    Ported from OLD_PWV_FrequencyFitGPAndMerra2ComparisonMarch2025.ipynb.

    Parameters
    ----------
    df_pairs      : pd.DataFrame — must contain columns 'dt' and 'PwvixPwvj'.
    list_of_bins  : array-like  — edges of the lag bins (any unit).
    averPWV       : float       — mean PWV (unused but kept for API compatibility).
    sigPWV        : float       — sigma PWV used for normalisation: C = <PwvixPwvj> / sigPWV^2.

    Returns
    -------
    xcenter  : np.ndarray — bin centres (arithmetic mean of edges).
    ydata    : np.ndarray — normalised mean product C(Dt).
    ydataerr : np.ndarray — standard error on C(Dt) in each bin.
    ncount   : np.ndarray — number of pairs in each bin.
    """
    Nbins   = len(list_of_bins)
    xcenter = (list_of_bins[:-1] + list_of_bins[1:]) / 2.0
    N       = len(xcenter)
    ydata    = np.zeros(N)
    ydataerr = np.zeros(N)
    ncount   = np.zeros(N, dtype=int)

    for ibin in range(Nbins - 1):
        cut    = (df_pairs['dt'] >= list_of_bins[ibin]) & \
                 (df_pairs['dt'] <  list_of_bins[ibin + 1])
        df_sel = df_pairs[cut]

        all_y = df_sel['PwvixPwvj'] / sigPWV**2

        n = len(all_y)
        ncount[ibin]   = n
        ydata[ibin]    = np.mean(all_y) if n > 0 else np.nan
        ydataerr[ibin] = (np.std(all_y) / np.sqrt(n)) if n > 1 else np.nan

    return xcenter, ydata, ydataerr, ncount


---
### 4a. Run Sylvie's estimator — same-night pairs


In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
LAG_MIN_MIN = 1.0     # minutes
LAG_MAX_MIN = 600.0   # 10 hours
N_BINS_LOG  = 50      # log-spaced bins
MIN_PAIRS   = 5       # minimum pairs per bin for display

# ── Log-spaced bin edges ──────────────────────────────────────────────────────
bin_edges = np.logspace(
    np.log10(LAG_MIN_MIN),
    np.log10(LAG_MAX_MIN),
    N_BINS_LOG + 1
)

# ── Build pair dataframe (same-night only) ────────────────────────────────────
print('Computing same-night pairs...')
df_pairs, pwv_var = ComputePWVAndTimeDifference_SameNight(
    df_keep, meanpwv_col='dPWV', t_col='t_min',
    night_col='nightObs'
)

print(f'Total same-night pairs in (0, 600] min : {len(df_pairs):,}')
print(f'Pooled intra-night sigma^2             : {pwv_var:.4f} mm^2')
print(f'                         sigma         : {np.sqrt(pwv_var):.4f} mm')

# Keep only pairs within the lag window
df_pairs = df_pairs[(df_pairs['dt'] >= LAG_MIN_MIN) & (df_pairs['dt'] <= LAG_MAX_MIN)]
print(f'Pairs after lag cut [{LAG_MIN_MIN:.0f}, {LAG_MAX_MIN:.0f}] min : {len(df_pairs):,}')


In [ ]:
# ── Run DCF computation ────────────────────────────────────────────────────────
sigma_pwv = np.sqrt(pwv_var)

xcenter, corr, corr_err, ncount = ComputeMyDCF_PwvixPwvj(
    df_pairs, bin_edges,
    averPWV=df_keep['dPWV'].mean(),
    sigPWV=sigma_pwv
)

# Geometric centre of log-spaced bins (matches PWV03 convention)
lag_min     = np.sqrt(bin_edges[:-1] * bin_edges[1:])   # geometric mean
lag_hr      = lag_min / 60.0

good = ncount >= MIN_PAIRS
print(f'Bins with >= {MIN_PAIRS} pairs : {good.sum()} / {N_BINS_LOG}')

# ── 1/e crossing ──────────────────────────────────────────────────────────────
inv_e   = 1.0 / np.e
cross   = np.where(good & (corr < inv_e))[0]
if len(cross) > 0:
    tau_min_1e = lag_min[cross[0]]
    tau_hr_1e  = lag_hr[cross[0]]
    print(f'Decorrelation timescale (C < 1/e) : {tau_min_1e:.1f} min = {tau_hr_1e:.2f} h')
else:
    tau_min_1e = np.nan
    tau_hr_1e  = np.nan
    print('C(Dt) does not drop below 1/e within 10 h')


---
## 5. Main figure — 2-panel (linear hours | log minutes)


In [ ]:
CORR_COLOR  = 'steelblue'
ZERO_COLOR  = 'gray'
INV_E_COLOR = 'darkorange'
TAU_COLOR   = 'darkred'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.subplots_adjust(wspace=0.32)

for ax, xdata, tau_x, xlabel, xscale, xlim in [
    (axes[0], lag_hr[good],  tau_hr_1e,  r'$\Delta t$ [hours]',   'linear', (0, LAG_MAX_MIN/60)),
    (axes[1], lag_min[good], tau_min_1e, r'$\Delta t$ [minutes]', 'log',    (LAG_MIN_MIN, LAG_MAX_MIN)),
]:
    ax.fill_between(
        xdata,
        corr[good] - corr_err[good],
        corr[good] + corr_err[good],
        alpha=0.25, color=CORR_COLOR
    )
    ax.plot(xdata, corr[good],
            color=CORR_COLOR, lw=1.8, marker='o', ms=3.5, zorder=3,
            label=r'$C(\Delta t)$  [Sylvie]')
    ax.axhline(0.0,  ls='--', color=ZERO_COLOR,  lw=1.0)
    ax.axhline(inv_e, ls=':', color=INV_E_COLOR, lw=1.5, label=r'$C = 1/e$')
    if np.isfinite(tau_x):
        ax.axvline(tau_x, ls='-', color=TAU_COLOR, lw=1.5,
                   label=rf'$\tau_{{1/e}} = {tau_x:.1f}$')
    ax.set_xscale(xscale)
    ax.set_xlim(xlim)
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_ylabel(r'$C(\Delta t)$', fontsize=13)
    ax.legend(fontsize=9, loc='upper right')

axes[0].set_title('Sylvie estimator — linear hours', fontsize=11)
axes[1].set_title('Sylvie estimator — log minutes',  fontsize=11)
fig.suptitle(f'PWV two-point correlation [Sylvie] — {version_run}', fontsize=12, y=1.01)

figfile = f'{pathfigs}/{prefix}_{version_run}_sylvie_main.pdf'
fig.savefig(figfile, bbox_inches='tight')
print(f'Saved: {figfile}')
plt.show()


---
## 6. Pair count per bin


In [ ]:
fig3, ax3 = plt.subplots(figsize=(12, 4))
ax3.bar(np.arange(N_BINS_LOG)[good], ncount[good],
        width=0.8, color='steelblue', alpha=0.8,
        edgecolor='white', linewidth=0.4)
ax3.set_yscale('log')
ax3.set_xlabel('Bin index', fontsize=12)
ax3.set_ylabel('Number of same-night pairs', fontsize=12)
ax3.set_title('Pair count per log-lag bin  (1 min - 10 h, same-night only)', fontsize=11)
tick_idx = np.linspace(0, N_BINS_LOG - 1, 8, dtype=int)
ax3.set_xticks(tick_idx)
ax3.set_xticklabels(
    [f'{lag_min[k]:.1f} min' if lag_min[k] < 60 else f'{lag_hr[k]:.1f} h'
     for k in tick_idx],
    rotation=30, ha='right', fontsize=9
)
fig3.tight_layout()
figfile3 = f'{pathfigs}/{prefix}_{version_run}_pair_counts.pdf'
fig3.savefig(figfile3, bbox_inches='tight')
print(f'Saved: {figfile3}')
plt.show()


---
## 7. Exponential fit of the decorrelation timescale


In [ ]:
def exp_decay(dt, A, tau):
    return A * np.exp(-dt / tau)

fit_mask = (
    good
    & (lag_min <= LAG_MAX_MIN)
    & (corr > 0)
    & np.isfinite(corr)
    & np.isfinite(corr_err)
    & (corr_err > 0)
)

x_fit = lag_min[fit_mask]
y_fit = corr[fit_mask]
e_fit = corr_err[fit_mask]

tau_guess = tau_min_1e if np.isfinite(tau_min_1e) else 60.0

try:
    popt, pcov = curve_fit(
        exp_decay, x_fit, y_fit,
        p0=[1.0, tau_guess],
        sigma=e_fit, absolute_sigma=True,
        bounds=([0, 0.1], [2.0, LAG_MAX_MIN]),
        maxfev=20_000
    )
    perr = np.sqrt(np.diag(pcov))
    A_fit, tau_fit_min = popt
    A_err, tau_fit_err = perr
    tau_fit_hr = tau_fit_min / 60.0
    print('=== Exponential fit  C(dt) = A * exp(-dt/tau) ===')
    print(f'  A   = {A_fit:.4f} +/- {A_err:.4f}')
    print(f'  tau = {tau_fit_min:.1f} +/- {tau_fit_err:.1f} min')
    print(f'      = {tau_fit_hr:.2f} +/- {tau_fit_err/60:.2f} h')
    fit_ok = True
except Exception as exc:
    print(f'Fit failed: {exc}')
    fit_ok = False


---
## 8. Per-filter correlation


In [ ]:
FILTERS_OF_INTEREST = ['empty', 'OG550_65mm_1', 'FELH0600']
filter_color = {
    'empty'        : '#1f77b4',
    'OG550_65mm_1' : '#d62728',
    'FELH0600'     : '#2ca02c',
}
filter_label = {
    'empty'        : 'empty',
    'OG550_65mm_1' : 'OG550',
    'FELH0600'     : 'FELH0600',
}

corr_by_filter     = {}
corr_err_by_filter = {}
ncount_by_filter   = {}

for filt in FILTERS_OF_INTEREST:
    sub_f = df_keep[df_keep['FILTER'] == filt].copy()
    if len(sub_f) < 10:
        print(f'  {filt}: too few points ({len(sub_f)}), skipped')
        continue
    # Per-filter nightly mean subtraction
    sub_f['dPWV_f'] = sub_f['PWV'] - sub_f.groupby('nightObs')['PWV'].transform('mean')
    var_f = sub_f['dPWV_f'].var(ddof=1) + SIGMA_repeat**2

    sub_f = sub_f.copy()
    sub_f['dPWV_use'] = sub_f['dPWV_f']
    df_pairs_f, _ = ComputePWVAndTimeDifference_SameNight(
        sub_f, meanpwv_col='dPWV_use', t_col='t_min',
        night_col='nightObs', sigma2=var_f
    )
    df_pairs_f = df_pairs_f[
        (df_pairs_f['dt'] >= LAG_MIN_MIN) & (df_pairs_f['dt'] <= LAG_MAX_MIN)
    ]
    if len(df_pairs_f) == 0:
        continue

    xc_f, corr_f, cerr_f, nc_f = ComputeMyDCF_PwvixPwvj(
        df_pairs_f, bin_edges,
        averPWV=sub_f['dPWV_f'].mean(),
        sigPWV=np.sqrt(var_f)
    )
    corr_by_filter[filt]     = corr_f
    corr_err_by_filter[filt] = cerr_f
    ncount_by_filter[filt]   = nc_f
    print(f'  {filt}: {nc_f.sum():,} pairs, {(nc_f >= MIN_PAIRS).sum()} good bins')


In [ ]:
fig4, axes4 = plt.subplots(1, 2, figsize=(16, 6))
fig4.subplots_adjust(wspace=0.32)

for ax4, xdata, xlabel, xscale, xlim in [
    (axes4[0], lag_hr,  r'$\Delta t$ [hours]',   'linear', (0, LAG_MAX_MIN/60)),
    (axes4[1], lag_min, r'$\Delta t$ [minutes]',  'log',    (LAG_MIN_MIN, LAG_MAX_MIN)),
]:
    ax4.plot(xdata[good], corr[good],
             color='black', lw=1.0, alpha=0.4, ls='--', label='All filters')
    for filt in FILTERS_OF_INTEREST:
        if filt not in corr_by_filter:
            continue
        c_f  = corr_by_filter[filt]
        ce_f = corr_err_by_filter[filt]
        gf   = ncount_by_filter[filt] >= MIN_PAIRS
        ax4.fill_between(xdata[gf], c_f[gf] - ce_f[gf], c_f[gf] + ce_f[gf],
                         alpha=0.15, color=filter_color[filt])
        ax4.plot(xdata[gf], c_f[gf],
                 color=filter_color[filt], lw=1.8, marker='o', ms=3,
                 label=filter_label[filt])
    ax4.axhline(0.0,  ls='--', color=ZERO_COLOR,  lw=1.0)
    ax4.axhline(inv_e, ls=':', color=INV_E_COLOR, lw=1.5, label=r'$C=1/e$')
    ax4.set_xscale(xscale)
    ax4.set_xlim(xlim)
    ax4.set_ylim(-1.35, 1.35)
    ax4.set_xlabel(xlabel, fontsize=13)
    ax4.set_ylabel(r'$C(\Delta t)$', fontsize=13)
    ax4.legend(fontsize=9, loc='upper right')

axes4[0].set_title('Per-filter correlation [Sylvie] — linear hours', fontsize=11)
axes4[1].set_title('Per-filter correlation [Sylvie] — log minutes',  fontsize=11)
fig4.suptitle(f'PWV two-point correlation by filter [Sylvie] — {version_run}', fontsize=12, y=1.01)
figfile4 = f'{pathfigs}/{prefix}_{version_run}_sylvie_per_filter.pdf'
fig4.savefig(figfile4, bbox_inches='tight')
print(f'Saved: {figfile4}')
plt.show()


---
## 9. Summary


In [ ]:
print('=== PWV two-point correlation [Sylvie estimator] summary ===')
print(f'  N observations (after cuts)  : {len(df_keep)}')
print(f'  N nights (nightObs)          : {df_keep["nightObs"].nunique()}')
print(f'  Same-night pairs (all bins)  : {len(df_pairs):,}')
print(f'  Pooled sigma                 : {sigma_pwv:.3f} mm')
print()
if np.isfinite(tau_min_1e):
    print(f'  1/e crossing (empirical): {tau_min_1e:.1f} min = {tau_hr_1e:.2f} h')
else:
    print('  C(Dt) does not drop below 1/e within 10 h')
if fit_ok:
    print(f'  Exp-fit tau : {tau_fit_min:.1f} +/- {tau_fit_err:.1f} min')
    print(f'              : {tau_fit_hr:.2f} +/- {tau_fit_err/60:.2f} h')
